# Important: sample code, safety warning, and license

## Safety warning

This notebook is sample code provided **AS IS**, without warranties, subject to the MIT License below. A license disclaimer does not guarantee immunity from liability; applicable law may limit its effect.

- **This notebook moves files and may overwrite existing destination files.** Test it in a non-production environment, maintain backups, and verify source and destination paths before running it.
- It updates event-processing records. Confirm permissions, configuration, and recovery procedures before using it with production data.
- File-signature matching is **not malware detection** and does not guarantee that an approved file is safe. Matching is exact and case-sensitive; unexpected signature output can cause an otherwise intended file type to be quarantined.
- Configure and enforce access controls separately. Moving a file into a folder named `quarantine` does not by itself restrict access or isolate malicious content.
- Running or saving this notebook may add outputs and environment metadata. Review and clear these before sharing; do not publish credentials, tokens, personal data, or private resource identifiers.

## License and third-party dependencies

Copyright (c) 2026 COPYRIGHT HOLDER

This notebook's original code is licensed under the **MIT License**; see the accompanying [LICENSE](./LICENSE) file for the full permission notice, warranty disclaimer, and limitation of liability. Include that file when redistributing the notebook.

Third-party packages and Microsoft Fabric services remain subject to their respective licenses and terms. This MIT notice does not relicense those dependencies. Review applicable licenses and retain any required third-party notices when redistributing dependencies.

In [ ]:
# Pre-req
# Configure pipeline in Real Time Hub where Storage Events are written to a lakehouse
# Setup a dedicated lakehouse / tables to store events. Consider using the same LH to store data for all workspaces storage audit logs.
# Setup quarantine folder to review / pass files - this should be accessible to admins only.
# For each user workspace, allow users to upload files to specific locations such as /Files/<user>. All other folder access must be blocked using onelake sec readwrite.
# - This will act as a guardrail to prevent users to moving files around.
# - Setup a framework and give users flexibility to upload files.
# - Configure folder which users can write to.

In [ ]:
# https://pypi.org/project/python-magic/
# python-magic is a Python interface to the libmagic file type identification library. libmagic identifies file types by checking their headers according to a predefined list of file types. 
# This functionality is exposed to the command line by the Unix command file.

# Alternatively, add python-magic as a PyPI library in a Fabric Spark Environment,
# publish the environment, and attach it to this notebook. You can then skip the
# install command below instead of installing the package on every notebook run.
# https://pypi.org/project/python-magic/
!pip install python-magic

In [ ]:
import duckdb
import magic
import fsspec
from deltalake import DeltaTable
import traceback
from datetime import datetime

In [ ]:
# Use this cell to configure input variables

# Workspace/folder to scan
prefix = "https://onelake.dfs.fabric.microsoft.com/"
user_workspace_id = "<replace with workspace id where users upload files>"
user_lakehouse_id = "<replace with lakehouse id where users upload files>"
scan_folder_name = "<replace with the folder which users upload files to>"

# This notebook uses folder named quarantine to move files which do not match allowed formats.
# You can use a different name depending on your requirement.
quarantine_folder = "quarantine"

# This notebook uses folder named ready_for_processing to move files which are ready for further processing.
# You can use a different name depending on your requirement.
ready_for_processing = "ready_for_processing" 


# Move files to quarantine section belonging to the workspace.
# This is the full abfss path. Later in the notebook, this abfss path will be passed to MS notebook utils to move the file to quarantine.
quarantine_path = f"abfss://{user_workspace_id}@onelake.dfs.fabric.microsoft.com/{user_lakehouse_id}/Files/{quarantine_folder}"

# Move files which are ready for processing to /ready_to_processing folder.
# This is full abfss path.
ready_for_processing_path = f"abfss://{user_workspace_id}@onelake.dfs.fabric.microsoft.com/{user_lakehouse_id}/Files/{ready_for_processing}"


# Workspace/folder details for admin
admin_workspace_id = "<replace with admin workspace id>"
admin_lakehouse_id = "<replace with admin lakehouse id>"
lh_events_table_schema = "<replace with schema name for schema where events table resides>"
lh_events_table_name = "<replace with events table name>"
events_table_path = f"abfss://{admin_workspace_id}@onelake.dfs.fabric.microsoft.com/{admin_lakehouse_id}/Tables/{lh_events_table_schema}/{lh_events_table_name}"

# To mark that the file captured in an event has been scanned or processed, we use the value = 1
# When this notebook runs, it uses SELECT * .. WHERE event_processed <> 1 to identify files which need to be scanned.
# At the end of the scan, this notebook will update event_processed column in the events table to 1 so that when you re-run the script, same record is NOT processed again
event_processed_flag = 1

# Error logging 
error_log_folder = "error_logs"
error_log_path = f"abfss://{admin_workspace_id}@onelake.dfs.fabric.microsoft.com/{admin_lakehouse_id}/Files/{error_log_folder}"


# Allowed formats use an exact, case-sensitive string match against the magic output.
# For each approved file type, scan a representative file and copy the full magic
# output exactly as returned into this list, including spaces and punctuation.
# Any output not exactly matching an entry is quarantined, even for an intended
# approved file type (e.g., 'ASCII text, with CRLF line terminators' != 'ASCII text').
allowed_formats = ['ASCII text','Apache Parquet']

In [ ]:
# Error logging configuration
#error_log_folder = "/Files/error_logs/"
#error_log_path = "abfss://"+sec_workspace_id+"@onelake.dfs.fabric.microsoft.com/"+sec_lakehouse_id+error_log_folder

def log_error_to_onelake(error_message: str, error_type: str, context: str = "") -> None:
    """
    Log errors to Fabric OneLake for administrator review.
    
    Args:
        error_message: The error message or traceback
        error_type: Type of error (e.g., 'FileProcessingError', 'ConfigurationError')
        context: Additional context about where the error occurred
    """
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        log_filename = f"error_{timestamp}_{error_type}.txt"
        log_file_path = error_log_path + log_filename
        
        log_content = f"""
{'='*80}
ERROR LOG
{'='*80}
Timestamp: {datetime.now().isoformat()}
Error Type: {error_type}
Context: {context}
{'='*80}
ERROR DETAILS:
{'='*80}
{error_message}
{'='*80}
"""
        
        # Create error log directory if it doesn't exist
        try:
            notebookutils.fs.mkdirs(error_log_path)
        except:
            pass  # Directory might already exist
        
        # Write error log to OneLake
        with onelake_fs.open(log_file_path, 'w') as f:
            f.write(log_content)
        
        print(f"Error logged to: {log_file_path}")
        
    except Exception as e:
        # Fallback: print to console if logging fails
        print(f"CRITICAL: Failed to log error to OneLake: {str(e)}")
#        print(f"Original error: {error_message}")

In [ ]:
# Run DuckDB query to fetch list of events which have event_processed = NULL
# NULL indicates that file signature for that record hasn't been validated.

query = f"""
        SELECT data['url']
        FROM delta_scan('{events_table_path}')
        WHERE event_processed is NULL
    """
urls_to_scan = duckdb.sql(query).fetchall()

In [ ]:
try:
    filesystem_code = "abfss"
    storage_options = {
        "account_name": "onelake",
        "account_host": "onelake.dfs.fabric.microsoft.com",
        "bearer_token": notebookutils.credentials.getToken('storage')
        }
        
    onelake_fs = fsspec.filesystem(filesystem_code, **storage_options)
    print("Successfully initialized OneLake filesystem")
    
except Exception as e:
    error_msg = f"Failed to initialize OneLake filesystem:\n{traceback.format_exc()}"
    log_error_to_onelake(error_msg, "FilesystemInitError", "OneLake connection setup")
    print(f"ERROR: {str(e)}")
    raise

In [ ]:
# Function to scan and determine file signature using Magic

def path_to_scan(abfss_path: str) -> str:
    """
    Scan file and determine its signature
    
    Args:
        abfss_path: The ABFSS path to the file
        
    Returns:
        File signature string
    """
    try:
        print("Scanning abfss path " + abfss_path)
        file_obj = onelake_fs.open(abfss_path, "rb").read()
        output = magic.Magic(uncompress=True).from_buffer(file_obj)
        print(f"File signature detected: {output}")
        return output
    except Exception as e:
        error_msg = f"Failed to scan file:\n{traceback.format_exc()}"
        log_error_to_onelake(error_msg, "FileScanError", f"path={abfss_path}")
        print(f"ERROR: Failed to scan file - {str(e)}")
        raise

In [ ]:
# Function to move files to quarantine or ready-for-processing storage

def move_file(abfss_path: str, target_path: str) -> str:
    """
    Move a file to a target folder, creating the path and overwriting if needed.
    
    Args:
        abfss_path: Source file ABFSS path
        target_path: Destination folder (quarantine_path or ready_for_processing_path)
        
    Returns:
        Status message
    """
    try:
        print(f"Moving file: {abfss_path} to: {target_path}")
            
        if not notebookutils.fs.mv(abfss_path, target_path, True, True):
            raise RuntimeError(f"File move returned False: {abfss_path} -> {target_path}")
        print(f"Successfully moved file to: {target_path}")
        return f"Success: File moved to {target_path}"
        
    except Exception as e:
        error_msg = f"Failed to move file:\n{traceback.format_exc()}"
        log_error_to_onelake(error_msg, "FileMoveError", f"source={abfss_path}, dest={target_path}")
        print(f"ERROR: Failed to move file - {str(e)}")
        raise

In [ ]:
# Function to construct abfss path from url extracted from OneLake events payload

def construct_abfss(input_url: str) -> tuple:
    """
    Construct ABFSS path from OneLake URL
    
    Args:
        input_url: OneLake URL from event
        
    Returns:
        Tuple of (file_abfss_path, workspace_id, item_path)
    """
    try:
        str_ver = "".join(input_url)
        rest_of_the_str = str_ver[len(prefix):]
        parts = rest_of_the_str.split("/", 2)
        
        if len(parts) < 3:
            raise ValueError(f"Invalid URL format: {input_url}")
            
        item_path = f"{parts[1]}/{parts[2]}"
        workspace_id = parts[0]
        file_abfss_path = "abfss://" + workspace_id + "@onelake.dfs.fabric.microsoft.com/" + item_path
        print("reconstructed abfss path is: " + file_abfss_path)
        return file_abfss_path, workspace_id, item_path
    except IndexError as e:
        error_msg = f"Invalid URL structure:\n{traceback.format_exc()}"
        log_error_to_onelake(error_msg, "URLParseError", f"input_url={input_url}")
        print(f"ERROR: Cannot parse URL - {input_url}")
        raise
    except Exception as e:
        error_msg = f"Failed to construct ABFSS path:\n{traceback.format_exc()}"
        log_error_to_onelake(error_msg, "ABFSSConstructionError", f"input_url={input_url}")
        print(f"ERROR: Failed to construct ABFSS path - {str(e)}")
        raise

In [ ]:
# Pending events: 1 = checked (allowed or quarantined), 99 = processing error.
def update_event_processed(event_url: str, status: int) -> None:
    escaped_url = event_url.replace("'", "''")
    events_table = DeltaTable(
        events_table_path,
        storage_options={
            "bearer_token": notebookutils.credentials.getToken("storage"),
            "use_fabric_endpoint": "true",
        },
    )
    result = events_table.update(
        updates={"event_processed": str(status)},
        predicate=f"data['url'] = '{escaped_url}' AND event_processed IS NULL",
    )
    if result["num_updated_rows"] == 0:
        raise RuntimeError(f"No pending event records updated for URL: {event_url}")
    print(f"Set event_processed = {status} for {result['num_updated_rows']} record(s).")


# Scan each URL once; the update marks all pending events for that URL.
urls_to_scan = list(dict.fromkeys(urls_to_scan))
processed_count = 0
error_count = 0
quarantined_count = 0
ready_for_processing_count = 0
status_update_error_count = 0
quarantine_error_count = 0

for idx, line in enumerate(urls_to_scan, 1):
    file_abfss_path = None
    # quarantine_destination = None
    file_quarantined = False
    try:
        print(f"\n{'='*60}")
        print(f"Processing file {idx}/{len(urls_to_scan)}")
        print(f"{'='*60}")
        
        file_abfss_path, workspace_id, item_path = construct_abfss(line)
        # quarantine_destination = quarantine_path.rstrip("/") + "/" + item_path.lstrip("/")
        print("full abfss path is: " + file_abfss_path)
        print("workspace id is: " + workspace_id)
        print("item_path is: " + item_path)
        
        print("checking file signature")
        file_sig = path_to_scan(file_abfss_path)
        print(file_sig)
        
        if file_sig in allowed_formats:
            print("File sig in allowed list")
            move_file(file_abfss_path, ready_for_processing_path)
            ready_for_processing_count += 1
            print("File moved to ready-for-processing storage.")
        else:
            print("File sig not in allowed list.")
            move_file(file_abfss_path, quarantine_path)
            file_quarantined = True
            quarantined_count += 1
            print("Quarantining file for further review. Please contact workspace administrator for next steps.")

        update_event_processed(line[0], 1)
        processed_count += 1
            
    except Exception as e:
        error_count += 1
        error_msg = f"Failed to process file at index {idx}:\nURL: {line}\n{traceback.format_exc()}"
        log_error_to_onelake(error_msg, "FileProcessingError", f"index={idx}, url={line}")
        print(f"ERROR: Failed to process file {idx} - {str(e)}")

        # Attempt both actions independently so one failure does not prevent the other.
        try:
            update_event_processed(line[0], 99)
        except Exception as status_error:
            status_update_error_count += 1
            log_error_to_onelake(
                f"Failed to set event_processed = 99:\n{traceback.format_exc()}",
                "EventStatusUpdateError",
                f"index={idx}, url={line}",
            )
            print(f"ERROR: Could not mark file {idx} as failed - {status_error}")

        print("Continuing with next file...")
        continue

print(f"\n{'='*60}")
print("PROCESSING SUMMARY")
print(f"{'='*60}")
print(f"Total files processed: {processed_count}/{len(urls_to_scan)}")
print(f"Files quarantined: {quarantined_count}")
print(f"Files ready for processing: {ready_for_processing_count}")
print(f"Errors encountered: {error_count}")
print(f"Failed error-status updates: {status_update_error_count}")
print(f"Failed error quarantines: {quarantine_error_count}")
print(f"{'='*60}")

if error_count > 0:
    print(f"\n {error_count} errors occurred during processing.")
    print(f"Check error logs at: {error_log_path}")